### Loading saved model (ELU activation function) --> quantization didnt work

In [ ]:
from tensorflow import keras
model_path = "/Users/noshin/Desktop/AUS/Masters/Research/New_dataset/saved_models/model_freq_101.h5"
model = keras.models.load_model(model_path1)


In [ ]:
model.summary()

Model: "sequential_28"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_48 (Conv2D)          (None, 99, 2, 32)         128       
                                                                 
 batch_normalization_54 (Ba  (None, 99, 2, 32)         128       
 tchNormalization)                                               
                                                                 
 elu_69 (ELU)                (None, 99, 2, 32)         0         
                                                                 
 conv2d_49 (Conv2D)          (None, 97, 2, 32)         3104      
                                                                 
 batch_normalization_55 (Ba  (None, 97, 2, 32)         128       
 tchNormalization)                                               
                                                                 
 elu_70 (ELU)                (None, 97, 2, 32)       

In [ ]:
import tensorflow_model_optimization as tfmot
import tensorflow as tf
from tensorflow import keras

quantize_model = tfmot.quantization.keras.quantize_model
qat_model = quantize_model(model) 


RuntimeError: Layer elu_69:<class 'tf_keras.src.layers.activation.elu.ELU'> is not supported. You can quantize this layer by passing a `tfmot.quantization.keras.QuantizeConfig` instance to the `quantize_annotate_layer` API.

### Model with ReLU activation

In [12]:
import scipy.io
import random
import numpy as np

In [13]:
# Load the .mat file
CTF_class_mov = scipy.io.loadmat('../CTF_Class_mov_final.mat')
CTF_class_static = scipy.io.loadmat('../CTF_Class_static_final.mat')

CTF_corridor_mov = scipy.io.loadmat('../CTF_Corridor_mov_final.mat')
CTF_corridor_static = scipy.io.loadmat('../CTF_Corridor_static_final.mat')

CTF_lab_mov = scipy.io.loadmat('../CTF_Lab_mov_final.mat')
CTF_lab_static = scipy.io.loadmat('../CTF_Lab_static_final.mat')

CTF_SC_mov = scipy.io.loadmat('../CTF_SC_mov_final.mat')
CTF_SC_static = scipy.io.loadmat('../CTF_SC_static_final.mat')

In [14]:
# Access the 4th item in each loaded variable
#static
CTF_class_static_array = CTF_class_static[list(CTF_class_static.keys())[3]].T
CTF_corridor_static_array = CTF_corridor_static[list(CTF_corridor_static.keys())[3]].T
CTF_lab_static_array = CTF_lab_static[list(CTF_lab_static.keys())[3]].T
CTF_SC_static_array = CTF_SC_static[list(CTF_SC_static.keys())[3]].T

# mov
CTF_class_mov_array = CTF_class_mov[list(CTF_class_mov.keys())[3]].T
CTF_corridor_mov_array = CTF_corridor_mov[list(CTF_corridor_mov.keys())[3]].T
CTF_lab_mov_array = CTF_lab_mov[list(CTF_lab_mov.keys())[3]].T
CTF_SC_mov_array = CTF_SC_mov[list(CTF_SC_mov.keys())[3]].T

CTF_class_static_array.shape

(38800, 2, 101)

In [15]:
# Combine real and imaginary parts for the 4th item in each loaded variable using stack
CTF_class = np.concatenate((CTF_class_static_array, CTF_class_mov_array), axis=0)
CTF_corridor = np.concatenate((CTF_corridor_static_array, CTF_corridor_mov_array), axis=0)
CTF_lab = np.concatenate((CTF_lab_static_array, CTF_lab_mov_array), axis=0)
CTF_SC = np.concatenate((CTF_SC_static_array, CTF_SC_mov_array), axis=0)

In [16]:
CTF_SC.shape

(77600, 2, 101)

In [17]:
# Combine 145 training points randomly selected from each env dataset into a single training set array, 
# and combine 49 testing points randomly selected from each dataset into a single testing set array.

# Number of unique grid points
num_grid_points = 194*2 #388

# Number of measurements per grid point
num_measurements = 200

# Number of grid points to select for training
num_train_points = int(0.75 * num_grid_points)  # 291

# Initialize empty lists for training and test sets
train_set = []
test_set = []

# For each array (representing a different environment)
for array in [CTF_class, CTF_corridor, CTF_lab, CTF_SC]:
    # Reshape the array to separate the measurements for each grid point
    reshaped_array = array.reshape(num_grid_points, num_measurements, -1, 2)
    
    # Randomly select grid points for training
    train_points = random.sample(range(num_grid_points), num_train_points)
    #print(len(train_points))
        
    # Get boolean array for test points
    test_points_bool = ~np.isin(range(num_grid_points), train_points)

    # Print test points
    # test_points = np.arange(num_grid_points)[test_points_bool]
    # print("Test points len:", len(test_points.tolist()))
    
    # Add the selected grid points to the training set and the rest to the test set
    train_set.append(reshaped_array[train_points])
    test_set.append(reshaped_array[test_points_bool])
    
    # ~ is the logical NOT operator, so ~np.isin(shuffle_index, train_points) gives all the indices not in train_points.
# Concatenate the data from all environments
train_set = np.concatenate(train_set, axis=0)
test_set = np.concatenate(test_set, axis=0)

print("Training set:",train_set.shape)
print("Testing set:",test_set.shape)

Training set: (1164, 200, 101, 2)
Testing set: (388, 200, 101, 2)


In [18]:
# Reshaping the arrays
large_X_train = train_set.reshape([-1,101,2])
large_X_test = test_set.reshape([-1,101,2])
print("large_X_train after reshape:", large_X_train.shape)
print("large_X_test after reshape:", large_X_test.shape)

large_X_train after reshape: (232800, 101, 2)
large_X_test after reshape: (77600, 101, 2)


In [19]:
# Create Labels for training data
#static
label1 = np.ones([291*200]) * 0  # class
label2 = np.ones([291*200]) * 1  # corridor
label3 = np.ones([291*200]) * 2  # lab
label4 = np.ones([291*200]) * 3  # SC

large_Y_train = np.concatenate([label1, label2, label3, label4])

label5 = np.ones([97*200]) * 0  # class
label6 = np.ones([97*200]) * 1  # corridor
label7 = np.ones([97*200]) * 2  # lab
label8 = np.ones([97*200]) * 3  # SC

large_Y_test = np.concatenate([label5, label6, label7, label8])

print("Training labels:", large_Y_train.shape)
print("Testing labels:", large_Y_test.shape)

Training labels: (232800,)
Testing labels: (77600,)


In [20]:
# Shuffle Data
shuffle_index1 = random.sample(range(0,232800), 232800)
#print(shuffle_index1)

# Shuffle data using the shuffled indices
large_X_train_new = large_X_train[shuffle_index1, :, :]
print(shuffle_index1[232797:])

large_Y_train = large_Y_train[shuffle_index1]
large_Y_train

[79889, 24711, 69821]


array([1., 2., 0., ..., 1., 0., 1.])

In [21]:
# Shuffle testing data
shuffle_index2 = random.sample(range(0,77600), 77600)

# Shuffle testing data using the shuffled indices
large_X_test_new = large_X_test[shuffle_index2, :, :]
#print(shuffle_index2[0:3])
print(shuffle_index2[77597:])

large_Y_test = large_Y_test[shuffle_index2]
large_Y_test

[49367, 22525, 1069]


array([2., 1., 3., ..., 2., 1., 0.])

In [22]:
from keras.utils import to_categorical

# Converting labels to categorical one-hot encoding
y_train = to_categorical(large_Y_train, num_classes=4) # (232000,)
y_test = to_categorical(large_Y_test, num_classes=4) # (78400,)

X_train = large_X_train_new # (232000, 101, 2) 
X_test = large_X_test_new # (77600, 101, 2) 

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Flatten, Dropout, BatchNormalization, ReLU
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler

# Build model
model = Sequential([
    Conv2D(32, kernel_size=(3, 1), input_shape=(101, 2, 1), strides=(1,1)),
    BatchNormalization(),
    ReLU(),

    Conv2D(32, kernel_size=(3, 1)),
    BatchNormalization(),
    ReLU(),
    
    Flatten(),
    Dense(200),
    ReLU(),
    Dropout(0.4),
    Dense(4, activation='softmax')
])

def lr_schedule(epoch):
    lr = 0.001
    if epoch > 2:
        lr *= 0.5e-3
    elif epoch > 4:
        lr *= 1e-3
    return lr

lr_scheduler = LearningRateScheduler(lr_schedule)
optimizer = Adam(learning_rate=lr_schedule(0))

model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, callbacks=[lr_scheduler])


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
2633/7275 ━━━━━━━━━━━━━━━━━━━━ 43s 9ms/step - accuracy: 0.9517 - loss: 0.1278

In [ ]:
model.save("model_freq_101_relu.keras")

In [ ]:
# Save only the weights (very small)
model.save_weights("model_freq_101_relu_weights_only.h5")

In [30]:
from tensorflow import keras
model_path1 = "/Users/noshin/Desktop/AUS/Masters/Research/New_dataset/Quantization/model_freq_101_relu.h5"
model1 = keras.models.load_model(model_path1)


TypeError: Error when deserializing class 'InputLayer' using config={'batch_shape': [None, 101, 2, 1], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_2'}.

Exception encountered: Unrecognized keyword arguments: ['batch_shape']

In [10]:
model.summary()

Model: "sequential_28"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_48 (Conv2D)          (None, 99, 2, 32)         128       
                                                                 
 batch_normalization_54 (Ba  (None, 99, 2, 32)         128       
 tchNormalization)                                               
                                                                 
 elu_69 (ELU)                (None, 99, 2, 32)         0         
                                                                 
 conv2d_49 (Conv2D)          (None, 97, 2, 32)         3104      
                                                                 
 batch_normalization_55 (Ba  (None, 97, 2, 32)         128       
 tchNormalization)                                               
                                                                 
 elu_70 (ELU)                (None, 97, 2, 32)       

In [11]:
import tensorflow_model_optimization as tfmot
import tensorflow as tf
from tensorflow import keras

quantize_model = tfmot.quantization.keras.quantize_model
qat_model = quantize_model(model) 


RuntimeError: Layer elu_69:<class 'tf_keras.src.layers.activation.elu.ELU'> is not supported. You can quantize this layer by passing a `tfmot.quantization.keras.QuantizeConfig` instance to the `quantize_annotate_layer` API.

In [45]:
!python3 -m pip install --upgrade "tensorflow>=2.11,<2.16"
!python3 -m pip install --upgrade "tensorflow-model-optimization>=0.7.3"
!python3 -m pip uninstall -y keras keras-nightly keras2 keras2-nightly


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.7/205.7 MB 3.7 MB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 18.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.4.0
    Uninstalling urllib3-2.4.0:
      Successfully uninstalled urllib3-2.4.0
  Attempting uninstall: tensorflow-estimator━━━━ 0/6 [urllib3]
    Found existing installation: tensorflow-estimator 2.14.0m [urllib3]
    Uninstalling tensorflow-estimator-2.14.0: 0/6 [urllib3]
      Successfully uninstalled tensorflow-estimator-2.14.0 [urllib3]
  Attempting uninstall: ml-dtypes━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [tensorflow-estimator]
    Found existing installation: ml_dtypes 0.5.1━━━━━━━━━━━━━━ 1/6 [tensorflow-estimator]
    Uninstalling ml_dtypes-0.5.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [tensorflow-estimator]
      Successfully uninstalled ml_dtypes-0.5.1━━━━━━━

In [43]:
import tensorflow_model_optimization as tfmot
print("TensorFlow version:", tf.__version__)
print("TFMOT version:", tfmot.__version__)


ModuleNotFoundError: No module named 'tf_keras'

In [ ]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model1)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("model_freq_101_dynamic_quant.tflite", "wb") as f:
    f.write(tflite_model)

In [ ]:
import numpy as np
import tensorflow as tf

# Assume model1 is your loaded or trained Keras model
# and X_test is your test data of shape (N, 101, 2, 1)

def representative_data_gen(): 0.
    for i in range(100):  # 100 calibration samples
        data = X_test[i:i+1]
        yield [data.astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model1)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()

with open("model_freq_101_full_int8.tflite", "wb") as f:
    f.write(tflite_model_int8)

print("Quantized TFLite model (int8) exported successfully!")
